# Fine-Tuning MiniGPT-4 (InstructBLIP) on Kvasir-VQA-x1 — v2 Optimized

**Model:** `Salesforce/instructblip-flan-t5-xl`

**Metrics:** Accuracy, F1, BLEU-1/2/3/4, ROUGE-1/2/L, ECE

## 1. Install Dependencies
**After running → Runtime → Restart → skip to Cell 2.**

In [ ]:
!pip install -q --upgrade transformers>=4.40.0
!pip install -q datasets accelerate pillow pandas tqdm
!pip install -q peft bitsandbytes
!pip install -q nltk rouge-score matplotlib seaborn
print("\n" + "="*60 + "\n  RESTART RUNTIME NOW, then skip to Cell 2.\n" + "="*60)

## 2. Imports & Configuration
**Start here after restart.**

In [ ]:
import os, json, gc, math, re, time, warnings
warnings.filterwarnings('ignore')
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from PIL import Image
from tqdm.auto import tqdm
from transformers import (
    InstructBlipProcessor, InstructBlipForConditionalGeneration,
    BitsAndBytesConfig, get_scheduler
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.stem import PorterStemmer
from rouge_score import rouge_scorer
import matplotlib.pyplot as plt
import matplotlib, seaborn as sns
matplotlib.rcParams['figure.dpi'] = 120
matplotlib.rcParams['font.size'] = 11

# ===== CONFIGURATION =====
USE_DRIVE = True
MODEL_ID = 'Salesforce/instructblip-flan-t5-xl'
MODEL_NAME = 'MiniGPT-4 (InstructBLIP)'
MODEL_KEY = 'minigpt4_finetuned_v2'

MAX_TRAIN_SAMPLES = 2000
NUM_EPOCHS = 8
BATCH_SIZE = 2
GRAD_ACCUM = 8
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
Q_MAX_LEN = 128
A_MAX_LEN = 64
MAX_NEW_TOKENS = 64

LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.1
LORA_TARGETS = ['q', 'k', 'v', 'o']

NUM_EVAL_SAMPLES = 20

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/AI-ML-based-approaches-for-the-medical-sector'
else:
    PROJECT_DIR = '/content/medical-vqa'

DATA_DIR = os.path.join(PROJECT_DIR, 'data')
IMAGE_DIR = os.path.join(DATA_DIR, 'images')
RESULTS_DIR = os.path.join(PROJECT_DIR, 'results', 'predictions')
CKPT_DIR = os.path.join(PROJECT_DIR, 'checkpoints', 'minigpt4_lora_v2')
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem/1e9:.1f} GB')

## 3. Download Dataset

In [ ]:
if not USE_DRIVE:
    from datasets import load_dataset
    os.makedirs(IMAGE_DIR, exist_ok=True)
    ds_host = load_dataset('SimulaMet-HOST/Kvasir-VQA', split='raw')
    seen = set()
    for row in tqdm(ds_host, desc='Saving images'):
        if row['img_id'] not in seen:
            row['image'].save(os.path.join(IMAGE_DIR, f"{row['img_id']}.jpg"))
            seen.add(row['img_id'])
    for split in ['train', 'test']:
        ds = load_dataset('SimulaMet/Kvasir-VQA-x1', split=split)
        records = [{'img_id': r['img_id'], 'complexity': r['complexity'],
                    'question': r['question'], 'answer': r['answer'],
                    'question_class': r['question_class']} for r in ds]
        pd.DataFrame(records).to_csv(os.path.join(DATA_DIR, f'kvasir_vqa_x1_{split}.csv'), index=False)
else:
    print('Using data from Google Drive.')

## 4. Evaluation Metrics (Full Suite)

**10 metrics:** Accuracy, F1, BLEU-1, BLEU-2, BLEU-3, BLEU-4, ROUGE-1, ROUGE-2, ROUGE-L, ECE

In [ ]:
smoother = SmoothingFunction().method1
# ROUGE scorer with all three variants
rouge_sc = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
stm = PorterStemmer()

def normalize_text(text):
    text = text.strip().lower()
    text = re.sub(r'[^\w\s]', '', text)
    return [stm.stem(t) for t in text.split()]

def compute_word_f1(pred, gt):
    p = set(normalize_text(pred)); g = set(normalize_text(gt))
    if not p or not g: return 0.0
    c = p & g
    if not c: return 0.0
    pr = len(c)/len(p); rc = len(c)/len(g)
    return 2*pr*rc/(pr+rc)

def compute_bleu_n(pred, gt, n):
    """Compute BLEU-n (n=1,2,3,4) with stemmed tokens."""
    ref = normalize_text(gt); hyp = normalize_text(pred)
    if not ref or not hyp: return 0.0
    # BLEU-n weights: uniform over 1..n, zero for rest
    weights = tuple([1.0/n]*n + [0.0]*(4-n))
    try:
        return sentence_bleu([ref], hyp, weights=weights, smoothing_function=smoother)
    except:
        return 0.0

def compute_rouge_all(pred, gt):
    """Compute ROUGE-1, ROUGE-2, ROUGE-L F-measures."""
    if not pred.strip() or not gt.strip():
        return {'rouge1': 0.0, 'rouge2': 0.0, 'rougeL': 0.0}
    scores = rouge_sc.score(gt.strip().lower(), pred.strip().lower())
    return {k: scores[k].fmeasure for k in ['rouge1', 'rouge2', 'rougeL']}

def compute_ece(confs, accs, n_bins=10):
    if not confs: return 0.0
    bounds = np.linspace(0, 1, n_bins+1); ece = 0.0; n = len(confs)
    for i in range(n_bins):
        mask = [(bounds[i] <= c < bounds[i+1]) for c in confs]
        nb = sum(mask)
        if nb == 0: continue
        ece += (nb/n)*abs(np.mean([a for a,m in zip(accs,mask) if m]) - np.mean([c for c,m in zip(confs,mask) if m]))
    return ece

def select_diverse_samples(df, n, seed=42):
    samples = []
    for c in sorted(df['complexity'].unique()):
        sub = df[df['complexity']==c]
        samples.append(sub.sample(n=min(max(1,n//3),len(sub)), random_state=seed))
    return pd.concat(samples).head(n)

def evaluate_single(row, pred):
    gt = str(row['answer'])
    em = ' '.join(normalize_text(pred)) == ' '.join(normalize_text(gt))
    rouge_scores = compute_rouge_all(pred, gt)
    return {
        'img_id': row['img_id'], 'complexity': int(row['complexity']),
        'question_class': row['question_class'], 'question': row['question'],
        'ground_truth': gt, 'prediction': pred, 'exact_match': em,
        'word_f1': round(compute_word_f1(pred, gt), 3),
        'bleu_1': round(compute_bleu_n(pred, gt, 1), 3),
        'bleu_2': round(compute_bleu_n(pred, gt, 2), 3),
        'bleu_3': round(compute_bleu_n(pred, gt, 3), 3),
        'bleu_4': round(compute_bleu_n(pred, gt, 4), 3),
        'rouge_1': round(rouge_scores['rouge1'], 3),
        'rouge_2': round(rouge_scores['rouge2'], 3),
        'rouge_l': round(rouge_scores['rougeL'], 3),
    }

ALL_METRIC_KEYS = ['word_f1','bleu_1','bleu_2','bleu_3','bleu_4','rouge_1','rouge_2','rouge_l']

def compute_summary(results, name):
    if not results: return {'model': name, 'error': 'No results'}
    n = len(results)
    em = sum(1 for r in results if r['exact_match'])
    f1s = [r['word_f1'] for r in results]
    ece = compute_ece(f1s, [1.0 if r['exact_match'] else 0.0 for r in results])
    s = {'model': name, 'num_samples': n,
         'exact_match_accuracy': round(em/n*100, 1),
         'ece': round(ece*100, 2),
         'exact_matches': em, 'total': n}
    # Aggregate all metric averages
    for mk in ALL_METRIC_KEYS:
        s[f'avg_{mk}'] = round(np.mean([r[mk] for r in results])*100, 1)
    # Per-complexity breakdown
    rdf = pd.DataFrame(results); pc = {}
    for c in sorted(rdf['complexity'].unique()):
        cdf = rdf[rdf['complexity']==c]
        entry = {'exact_matches': int(cdf['exact_match'].sum()), 'total': int(len(cdf)),
                 'exact_accuracy': round(cdf['exact_match'].mean()*100, 1)}
        for mk in ALL_METRIC_KEYS:
            entry[f'avg_{mk}'] = round(cdf[mk].mean()*100, 1)
        pc[f'level_{c}'] = entry
    s['per_complexity'] = pc
    return s

def print_summary(s):
    print(f"\n{'='*70}\n  {s['model'].upper()} — RESULTS\n{'='*70}")
    print(f"  Accuracy:   {s.get('exact_match_accuracy',0):.1f}%")
    print(f"  F1:         {s.get('avg_word_f1',0):.1f}%")
    print(f"  BLEU-1:     {s.get('avg_bleu_1',0):.1f}%")
    print(f"  BLEU-2:     {s.get('avg_bleu_2',0):.1f}%")
    print(f"  BLEU-3:     {s.get('avg_bleu_3',0):.1f}%")
    print(f"  BLEU-4:     {s.get('avg_bleu_4',0):.1f}%")
    print(f"  ROUGE-1:    {s.get('avg_rouge_1',0):.1f}%")
    print(f"  ROUGE-2:    {s.get('avg_rouge_2',0):.1f}%")
    print(f"  ROUGE-L:    {s.get('avg_rouge_l',0):.1f}%")
    print(f"  ECE:        {s.get('ece',0):.2f}%")
    for k,v in s.get('per_complexity',{}).items():
        print(f"    {k}: EM {v['exact_matches']}/{v['total']}, F1 {v['avg_word_f1']:.1f}%, B1 {v['avg_bleu_1']:.1f}%")
    print('='*70)

print('Metrics loaded: Accuracy, F1, BLEU-1/2/3/4, ROUGE-1/2/L, ECE')

## 5. Prediction Grid

In [ ]:
def plot_prediction_grid(results, title, save_name, max_show=6):
    show = results[:max_show]; n = len(show)
    if n == 0: return
    cols = min(3, n); rows_n = math.ceil(n/cols)
    fig, axes = plt.subplots(rows_n, cols, figsize=(7*cols, 6*rows_n))
    if rows_n==1 and cols==1: axes = np.array([axes])
    axes = np.atleast_2d(axes)
    fig.suptitle(title, fontsize=16, fontweight='bold', y=1.01)
    for i, r in enumerate(show):
        ri, ci = divmod(i, cols); ax = axes[ri][ci]
        img_path = os.path.join(IMAGE_DIR, f"{r['img_id']}.jpg")
        if os.path.exists(img_path): ax.imshow(Image.open(img_path).convert('RGB'))
        ax.set_xticks([]); ax.set_yticks([])
        if r['exact_match']: st='EXACT ✓'; cl='#27ae60'
        elif r['word_f1']>=0.5: st='PARTIAL ~'; cl='#f39c12'
        else: st='WRONG ✗'; cl='#e74c3c'
        txt = f"{st} | F1:{r['word_f1']:.2f} B1:{r['bleu_1']:.2f} R1:{r['rouge_1']:.2f}\nQ: {r['question'][:90]}\nGT: {r['ground_truth'][:70]}\nPred: {r['prediction'][:70]}"
        ax.text(0.02, 0.98, txt, transform=ax.transAxes, fontsize=7, va='top', color=cl,
                fontweight='bold', bbox=dict(boxstyle='round,pad=0.3', facecolor='black', alpha=0.75))
    for i in range(n, rows_n*cols): axes[divmod(i,cols)[0]][divmod(i,cols)[1]].axis('off')
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, save_name), dpi=150, bbox_inches='tight')
    plt.show()
print('Grid loaded.')

## 6. Load Data

In [ ]:
train_df = pd.read_csv(os.path.join(DATA_DIR, 'kvasir_vqa_x1_train.csv'))
test_df = pd.read_csv(os.path.join(DATA_DIR, 'kvasir_vqa_x1_test.csv'))
train_df = train_df[train_df['img_id'].apply(lambda x: os.path.exists(os.path.join(IMAGE_DIR, f'{x}.jpg')))].reset_index(drop=True)
test_df = test_df[test_df['img_id'].apply(lambda x: os.path.exists(os.path.join(IMAGE_DIR, f'{x}.jpg')))].reset_index(drop=True)
if len(train_df) > MAX_TRAIN_SAMPLES:
    train_df = train_df.sample(n=MAX_TRAIN_SAMPLES, random_state=42).reset_index(drop=True)
eval_df = select_diverse_samples(test_df, NUM_EVAL_SAMPLES)
print(f'Training: {len(train_df)} | Test: {len(test_df)} | Eval: {len(eval_df)}')

## 7. Load Model (4-bit)

In [ ]:
print(f'[INFO] Loading {MODEL_ID}...')
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
processor = InstructBlipProcessor.from_pretrained(MODEL_ID)
model = InstructBlipForConditionalGeneration.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, device_map='auto')
model.eval()
if torch.cuda.is_available(): print(f'[INFO] GPU: {torch.cuda.memory_allocated()/1e9:.1f} GB')
print('[INFO] Loaded.')

## 8. Zero-Shot Evaluation

In [ ]:
def run_inference(model, processor, sample_df, label=''):
    results = []
    for idx, (_, row) in enumerate(sample_df.iterrows()):
        img = Image.open(os.path.join(IMAGE_DIR, f"{row['img_id']}.jpg")).convert('RGB')
        prompt = f"Answer this medical question concisely. Question: {row['question']} Answer:"
        try:
            inputs = processor(images=img, text=prompt, return_tensors='pt').to(model.device, torch.float16)
            with torch.inference_mode():
                out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
            pred = processor.decode(out[0], skip_special_tokens=True).strip()
        except Exception as e:
            print(f'[ERROR] {idx}: {e}'); pred = ''
        result = evaluate_single(row, pred)
        results.append(result)
        st = '✓' if result['exact_match'] else '~' if result['word_f1']>=0.5 else '✗'
        if idx < 5 or result['exact_match']:
            print(f"  [{idx+1}] {st} F1:{result['word_f1']:.2f} B1:{result['bleu_1']:.2f} R1:{result['rouge_1']:.2f} | {result['question'][:45]}")
    return results

print('=== ZERO-SHOT ===')
zs_results = run_inference(model, processor, eval_df)
zs_summary = compute_summary(zs_results, f'{MODEL_NAME} (Zero-Shot)')
print_summary(zs_summary)
pd.DataFrame(zs_results).to_csv(os.path.join(RESULTS_DIR, f'{MODEL_KEY}_zeroshot.csv'), index=False)

## 9. Zero-Shot Grid

In [ ]:
plot_prediction_grid(zs_results, f'{MODEL_NAME} — Zero-Shot', f'{MODEL_KEY}_zs_grid.png')

## 10. Training Dataset

In [ ]:
class KvasirVQADataset(Dataset):
    def __init__(self, df, processor, image_dir, q_max=128, a_max=64):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.image_dir = image_dir
        self.q_max = q_max; self.a_max = a_max
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(os.path.join(self.image_dir, f"{row['img_id']}.jpg")).convert('RGB')
        question = f"Answer this medical question concisely. Question: {row['question']} Answer:"
        answer = str(row['answer'])
        inputs = self.processor(images=img, text=question, return_tensors='pt',
            padding='max_length', max_length=self.q_max, truncation=True)
        labels = self.processor.tokenizer(answer, return_tensors='pt',
            padding='max_length', max_length=self.a_max, truncation=True).input_ids
        labels[labels == self.processor.tokenizer.pad_token_id] = -100
        item = {k: v.squeeze(0) for k, v in inputs.items()}
        item['labels'] = labels.squeeze(0)
        return item

def collate_fn(batch):
    return {k: torch.stack([b[k] for b in batch]) for k in batch[0].keys()}

train_dataset = KvasirVQADataset(train_df, processor, IMAGE_DIR, Q_MAX_LEN, A_MAX_LEN)
val_df = train_df.sample(n=min(100, len(train_df)), random_state=99)
val_dataset = KvasirVQADataset(val_df, processor, IMAGE_DIR, Q_MAX_LEN, A_MAX_LEN)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=0)
print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

## 11. Apply LoRA

In [ ]:
model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGETS, lora_dropout=LORA_DROPOUT,
    bias='none', task_type=TaskType.SEQ_2_SEQ_LM)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 12. Training Loop

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
total_steps = (len(train_loader) * NUM_EPOCHS) // GRAD_ACCUM
warmup_steps = int(total_steps * 0.1)
scheduler = get_scheduler('cosine', optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)

train_losses = []; val_losses = []
epoch_train_losses = []; epoch_val_losses = []
best_val_loss = float('inf')
patience = 3; no_improve = 0

print(f'\n{"="*70}')
print(f'  Epochs:{NUM_EPOCHS} Batch:{BATCH_SIZE} Accum:{GRAD_ACCUM} EffBatch:{BATCH_SIZE*GRAD_ACCUM}')
print(f'  LR:{LEARNING_RATE} Warmup:{warmup_steps} Steps:{total_steps} Scheduler:cosine')
print(f'  LoRA r={LORA_R} alpha={LORA_ALPHA} targets={LORA_TARGETS}')
print(f'{"="*70}\n')

model.train()
global_step = 0
start_time = time.time()

for epoch in range(NUM_EPOCHS):
    epoch_loss = 0.0; nb = 0
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS}')
    for step, batch in enumerate(pbar):
        batch = {k: v.to(model.device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss / GRAD_ACCUM
        loss.backward()
        epoch_loss += outputs.loss.item(); nb += 1
        if (step + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); scheduler.step(); optimizer.zero_grad()
            global_step += 1
            train_losses.append(outputs.loss.item())
        pbar.set_postfix({'loss': f'{outputs.loss.item():.4f}', 'lr': f'{scheduler.get_last_lr()[0]:.2e}'})

    avg_train = epoch_loss / max(nb, 1)
    epoch_train_losses.append(avg_train)

    model.eval()
    vl = 0.0; vb = 0
    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(model.device) for k, v in batch.items()}
            vl += model(**batch).loss.item(); vb += 1
    avg_val = vl / max(vb, 1)
    epoch_val_losses.append(avg_val)

    print(f'  Epoch {epoch+1}: Train={avg_train:.4f} Val={avg_val:.4f}')
    if avg_val < best_val_loss:
        best_val_loss = avg_val; no_improve = 0
        model.save_pretrained(CKPT_DIR)
        print(f'  ✓ Best saved (val={best_val_loss:.4f})')
    else:
        no_improve += 1
        if no_improve >= patience:
            print(f'  ⚠ Early stop at epoch {epoch+1}'); break
    model.train()

print(f'\nDone in {(time.time()-start_time)/60:.1f} min | Best val: {best_val_loss:.4f}')

## 13. Fine-Tuned Evaluation

In [ ]:
model.eval()
print('=== FINE-TUNED ===')
ft_results = run_inference(model, processor, eval_df)
ft_summary = compute_summary(ft_results, f'{MODEL_NAME} (Fine-Tuned v2)')
print_summary(ft_summary)
pd.DataFrame(ft_results).to_csv(os.path.join(RESULTS_DIR, f'{MODEL_KEY}_predictions.csv'), index=False)
with open(os.path.join(RESULTS_DIR, f'{MODEL_KEY}_summary.json'), 'w') as f:
    json.dump(ft_summary, f, indent=2)

## 14. Fine-Tuned Grid

In [ ]:
plot_prediction_grid(ft_results, f'{MODEL_NAME} — Fine-Tuned v2', f'{MODEL_KEY}_ft_grid.png')

## 15. 📊 Loss Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(train_losses, color='#3498db', alpha=0.4, linewidth=0.8, label='Step')
if len(train_losses) > 10:
    w = max(5, len(train_losses)//20)
    ax1.plot(pd.Series(train_losses).rolling(w, min_periods=1).mean(), color='#e74c3c', linewidth=2, label=f'Smooth(w={w})')
ax1.set_xlabel('Step'); ax1.set_ylabel('Loss'); ax1.set_title('Training Loss', fontweight='bold')
ax1.legend(); ax1.grid(True, alpha=0.3)
ep = range(1, len(epoch_train_losses)+1)
ax2.plot(ep, epoch_train_losses, 'o-', color='#3498db', linewidth=2, markersize=8, label='Train')
ax2.plot(ep, epoch_val_losses, 's-', color='#e74c3c', linewidth=2, markersize=8, label='Val')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss'); ax2.set_title('Train vs Val', fontweight='bold')
ax2.legend(); ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, f'{MODEL_KEY}_loss.png'), dpi=150, bbox_inches='tight')
plt.show()

## 16. 📊 Full Metrics Comparison (10 Metrics)

In [ ]:
# All 10 metrics comparison
metric_labels = ['Accuracy', 'F1', 'BLEU-1', 'BLEU-2', 'BLEU-3', 'BLEU-4', 'ROUGE-1', 'ROUGE-2', 'ROUGE-L', 'ECE']
metric_keys_zs = ['exact_match_accuracy','avg_word_f1','avg_bleu_1','avg_bleu_2','avg_bleu_3','avg_bleu_4','avg_rouge_1','avg_rouge_2','avg_rouge_l','ece']
zs_vals = [zs_summary.get(k, 0) for k in metric_keys_zs]
ft_vals = [ft_summary.get(k, 0) for k in metric_keys_zs]

x = np.arange(len(metric_labels)); width = 0.35
fig, ax = plt.subplots(figsize=(16, 6))
b1 = ax.bar(x - width/2, zs_vals, width, label='Zero-Shot', color='#95a5a6', edgecolor='white')
b2 = ax.bar(x + width/2, ft_vals, width, label='Fine-Tuned v2', color='#27ae60', edgecolor='white')
for b, v in zip(b1, zs_vals): ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.2, f'{v:.1f}', ha='center', fontsize=7, color='#666')
for b, v in zip(b2, ft_vals): ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.2, f'{v:.1f}', ha='center', fontsize=7, fontweight='bold')
ax.set_ylabel('Score (%)'); ax.set_title('Full Metrics: Zero-Shot vs Fine-Tuned', fontweight='bold', fontsize=14)
ax.set_xticks(x); ax.set_xticklabels(metric_labels, rotation=30, ha='right')
ax.legend(fontsize=11); ax.set_ylim(0, max(max(zs_vals+[1]), max(ft_vals+[1]))*1.3)
ax.grid(axis='y', alpha=0.3)
# Improvement arrows
for i, (z, f) in enumerate(zip(zs_vals, ft_vals)):
    d = f - z
    if i < 9:  # Higher is better
        c = '#27ae60' if d > 0 else '#e74c3c'; s = '▲' if d > 0 else '▼'
    else:  # ECE: lower is better
        c = '#27ae60' if d < 0 else '#e74c3c'; s = '▼' if d < 0 else '▲'
    ax.text(i, max(z,f)+1, f'{s}{abs(d):.1f}', ha='center', fontsize=6, color=c, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, f'{MODEL_KEY}_all_metrics.png'), dpi=150, bbox_inches='tight')
plt.show()

# Print table
print(f"\n{'Metric':<12} {'Zero-Shot':>10} {'Fine-Tuned':>12} {'Change':>10}")
print('-'*47)
for n, z, f in zip(metric_labels, zs_vals, ft_vals):
    print(f"{n:<12} {z:>9.1f}% {f:>11.1f}% {f-z:>+9.1f}pp")

## 17. 📊 BLEU & ROUGE Breakdown

In [ ]:
# Side-by-side BLEU and ROUGE breakdowns
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# BLEU breakdown
bleu_labels = ['BLEU-1', 'BLEU-2', 'BLEU-3', 'BLEU-4']
bleu_zs = [zs_summary.get(f'avg_bleu_{i}', 0) for i in range(1,5)]
bleu_ft = [ft_summary.get(f'avg_bleu_{i}', 0) for i in range(1,5)]
bx = np.arange(4); bw = 0.35
ax1.bar(bx-bw/2, bleu_zs, bw, label='Zero-Shot', color='#bdc3c7')
ax1.bar(bx+bw/2, bleu_ft, bw, label='Fine-Tuned', color='#2ecc71')
for i, (z, f) in enumerate(zip(bleu_zs, bleu_ft)):
    ax1.text(i-bw/2, z+0.3, f'{z:.1f}', ha='center', fontsize=8)
    ax1.text(i+bw/2, f+0.3, f'{f:.1f}', ha='center', fontsize=8, fontweight='bold')
ax1.set_xticks(bx); ax1.set_xticklabels(bleu_labels)
ax1.set_ylabel('Score (%)'); ax1.set_title('BLEU Scores Breakdown', fontweight='bold')
ax1.legend(); ax1.grid(axis='y', alpha=0.3)

# ROUGE breakdown
rouge_labels = ['ROUGE-1', 'ROUGE-2', 'ROUGE-L']
rouge_zs = [zs_summary.get(k, 0) for k in ['avg_rouge_1','avg_rouge_2','avg_rouge_l']]
rouge_ft = [ft_summary.get(k, 0) for k in ['avg_rouge_1','avg_rouge_2','avg_rouge_l']]
rx = np.arange(3); rw = 0.35
ax2.bar(rx-rw/2, rouge_zs, rw, label='Zero-Shot', color='#bdc3c7')
ax2.bar(rx+rw/2, rouge_ft, rw, label='Fine-Tuned', color='#e67e22')
for i, (z, f) in enumerate(zip(rouge_zs, rouge_ft)):
    ax2.text(i-rw/2, z+0.3, f'{z:.1f}', ha='center', fontsize=8)
    ax2.text(i+rw/2, f+0.3, f'{f:.1f}', ha='center', fontsize=8, fontweight='bold')
ax2.set_xticks(rx); ax2.set_xticklabels(rouge_labels)
ax2.set_ylabel('Score (%)'); ax2.set_title('ROUGE Scores Breakdown', fontweight='bold')
ax2.legend(); ax2.grid(axis='y', alpha=0.3)

plt.suptitle('Granular Text Metrics', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, f'{MODEL_KEY}_bleu_rouge.png'), dpi=150, bbox_inches='tight')
plt.show()

## 18. 📊 Calibration Plot

In [ ]:
def plot_calibration(results, title, ax):
    confs = [r['word_f1'] for r in results]
    accs = [1.0 if r['exact_match'] else 0.0 for r in results]
    bounds = np.linspace(0, 1, 11); bc = []; ba = []
    for i in range(10):
        mask = [(bounds[i] <= c < bounds[i+1]) for c in confs]; nb = sum(mask)
        bc.append(np.mean([c for c,m in zip(confs,mask) if m]) if nb > 0 else (bounds[i]+bounds[i+1])/2)
        ba.append(np.mean([a for a,m in zip(accs,mask) if m]) if nb > 0 else 0)
    ax.bar(range(10), ba, color='#3498db', alpha=0.7, label='Accuracy')
    ax.plot(range(10), bc, 'r--o', linewidth=2, markersize=5, label='Confidence')
    ax.set_title(title, fontweight='bold'); ax.legend(fontsize=8); ax.set_ylim(0, 1.1)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 5))
plot_calibration(zs_results, f'Zero-Shot (ECE={zs_summary.get("ece",0):.2f}%)', a1)
plot_calibration(ft_results, f'Fine-Tuned (ECE={ft_summary.get("ece",0):.2f}%)', a2)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, f'{MODEL_KEY}_calibration.png'), dpi=150, bbox_inches='tight')
plt.show()

## 19. 📊 Per-Complexity & Heatmap

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (res, nm, cl) in zip(axes, [(zs_results,'Zero-Shot','#95a5a6'),(ft_results,'Fine-Tuned','#27ae60')]):
    rdf = pd.DataFrame(res); lvls = sorted(rdf['complexity'].unique())
    f1 = [rdf[rdf['complexity']==l]['word_f1'].mean()*100 for l in lvls]
    em = [rdf[rdf['complexity']==l]['exact_match'].mean()*100 for l in lvls]
    xx = np.arange(len(lvls)); w = 0.35
    ax.bar(xx-w/2, em, w, label='Accuracy', color=cl, alpha=0.7)
    ax.bar(xx+w/2, f1, w, label='F1', color=cl)
    ax.set_xlabel('Complexity'); ax.set_ylabel('Score (%)')
    ax.set_title(f'{nm}', fontweight='bold')
    ax.set_xticks(xx); ax.set_xticklabels([f'L{l}' for l in lvls]); ax.legend(); ax.set_ylim(0,100)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, f'{MODEL_KEY}_complexity.png'), dpi=150, bbox_inches='tight')
plt.show()

ft_df = pd.DataFrame(ft_results)
if ft_df['question_class'].nunique() > 1:
    pv = ft_df.pivot_table(values='word_f1', index='question_class', columns='complexity', aggfunc='mean')*100
    fig, ax = plt.subplots(figsize=(8, max(4, len(pv)*0.5+1)))
    sns.heatmap(pv, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax, linewidths=0.5)
    ax.set_title('F1 by Class × Complexity', fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, f'{MODEL_KEY}_heatmap.png'), dpi=150, bbox_inches='tight')
    plt.show()

## 20. 📊 Side-by-Side Predictions

In [ ]:
ns = min(4, len(zs_results), len(ft_results))
fig, axes = plt.subplots(ns, 2, figsize=(14, 5*ns))
if ns == 1: axes = axes.reshape(1, -1)
fig.suptitle('Before vs After', fontsize=16, fontweight='bold', y=1.01)
for i in range(ns):
    for j, (r, lb) in enumerate([(zs_results[i], 'ZERO-SHOT'), (ft_results[i], 'FINE-TUNED')]):
        ax = axes[i][j]
        p = os.path.join(IMAGE_DIR, f"{r['img_id']}.jpg")
        if os.path.exists(p): ax.imshow(Image.open(p).convert('RGB'))
        ax.set_xticks([]); ax.set_yticks([])
        if r['exact_match']: st='✓'; cl='#27ae60'
        elif r['word_f1']>=0.5: st='~'; cl='#f39c12'
        else: st='✗'; cl='#e74c3c'
        t = f"[{lb}] {st} F1:{r['word_f1']:.2f} B1:{r['bleu_1']:.2f} R1:{r['rouge_1']:.2f}\nQ:{r['question'][:80]}\nGT:{r['ground_truth'][:60]}\nP:{r['prediction'][:60]}"
        ax.text(0.02, 0.98, t, transform=ax.transAxes, fontsize=7, va='top', color=cl,
                fontweight='bold', bbox=dict(boxstyle='round,pad=0.3', facecolor='black', alpha=0.75))
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, f'{MODEL_KEY}_before_after.png'), dpi=150, bbox_inches='tight')
plt.show()

## 21. Conclusion

In [ ]:
print(f"\n{'='*80}")
print(f'  RESULTS — {MODEL_NAME} (10 Metrics)')
print(f"{'='*80}")
print(f"  {'Metric':<12} {'Zero-Shot':>10} {'Fine-Tuned':>12} {'Δ':>8}")
print(f"  {'-'*45}")
improved = 0
for n, z, f in zip(metric_labels, zs_vals, ft_vals):
    d = f - z
    better = (d > 0 and n != 'ECE') or (d < 0 and n == 'ECE')
    a = '↑' if better else '↓'
    print(f"  {n:<12} {z:>9.1f}% {f:>11.1f}% {a}{abs(d):>6.1f}pp")
    if better: improved += 1
print(f"\n  {improved}/10 metrics improved.")
comparison = {'zero_shot': zs_summary, 'fine_tuned': ft_summary,
              'config': {'lr': LEARNING_RATE, 'epochs_run': len(epoch_train_losses),
                         'lora_r': LORA_R, 'lora_targets': LORA_TARGETS,
                         'train_samples': MAX_TRAIN_SAMPLES, 'best_val_loss': best_val_loss}}
with open(os.path.join(RESULTS_DIR, f'{MODEL_KEY}_comparison.json'), 'w') as f:
    json.dump(comparison, f, indent=2)
print(f"{'='*80}")
print(f'Saved to {RESULTS_DIR}')

## 22. Cleanup

In [ ]:
del model, processor
if torch.cuda.is_available(): torch.cuda.empty_cache()
gc.collect()
print('GPU cleared.')